<a href="https://colab.research.google.com/github/Shriyamaricharla/Data/blob/main/actual_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install pandas scikit-learn imbalanced-learn matplotlib interpret requests

# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
import numpy as np
from interpret.glassbox import (LogisticRegression, ExplainableBoostingClassifier, ClassificationTree)
from interpret.blackbox import LimeTabular
from sklearn.ensemble import RandomForestClassifier

from interpret import show
from sklearn.metrics import f1_score, accuracy_score
import requests
import pickle


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 38.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 14.9 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl size=40

In [2]:
# URL of the CSV file in the GitHub repository
#file_url = "https://raw.githubusercontent.com/marichala/ML/refs/heads/ExplainableAI/healthcare-dataset-stroke-data.csv"
file_url="https://raw.githubusercontent.com/Shriyamaricharla/Data/main/healthcare-dataset-stroke-data.csv"
# Load the CSV file into a pandas DataFrame
data = pd.read_csv(file_url)
data.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
categorical_cols = ["gender",
                            "ever_married",
                            "work_type",
                            "Residence_type",
                            "smoking_status"]

encoded = pd.get_dummies(data[categorical_cols], prefix=categorical_cols)
#encoded.info()

In [4]:
data = pd.concat([encoded, data], axis=1)
data.drop(categorical_cols, axis=1, inplace=True)

#data.info()#

# Impute missing values of BMI
data.bmi = data.bmi.fillna(0)

# Drop id as it is not relevant
if 'id' in data.columns:
  data.drop(['id'], axis=1, inplace=True)

#spliting features (X) and lables (y)
#Features = all columns in the dataset except the last column.... represented :(start default 0) to :-1 (minus 1 column from end)
X = data.iloc[:,:-1]
y = data.iloc[:,-1]

# Split the data for evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=2021)

print("Before oversampling:", X_train.shape)
oversample = RandomOverSampler(sampling_strategy='minority')

# Convert to numpy and oversample
x_np = X_train.to_numpy()
y_np = y_train.to_numpy()

# Convert y to int before oversampling
y_np = y_np.astype(int)

x_np, y_np = oversample.fit_resample(x_np, y_np)

# Convert boolean values to 1.0 or 0.0
x_np = np.where(x_np == True, 1, x_np)
x_np = np.where(x_np == False, 0, x_np)

y_np = np.where(y_np == True, 1, y_np)
y_np = np.where(y_np == False, 0, y_np)

# Convert back to pandas
X_train = pd.DataFrame(x_np, columns=X_train.columns)
y_train = pd.Series(y_np, name=y_train.name)

# Check for non-numeric values in each column
for col in X_train.columns:
    if not pd.api.types.is_numeric_dtype(X_train[col]):
        print(f"Train Column '{col}' contains non-numeric values.")
        # Handle non-numeric values (e.g., convert to numeric or remove rows)
        # Example: Convert to numeric, coercing errors to NaN
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        # Or remove rows with non-numeric values:
        # X_train = X_train[pd.to_numeric(X_train[col], errors='coerce').notnull()]

# Convert columns to float if all values are numeric
for col in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[col]):
        X_train[col] = X_train[col].astype(float)

# Check for non-numeric values in each column
for col in X_test.columns:
    if not pd.api.types.is_numeric_dtype(X_test[col]):
        print(f"Test Column '{col}' contains non-numeric values.")
        # Handle non-numeric values (e.g., convert to numeric or remove rows)
        # Example: Convert to numeric, coercing errors to NaN
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
        # Or remove rows with non-numeric values:
        # X_train = X_train[pd.to_numeric(X_train[col], errors='coerce').notnull()]

# Convert columns to float if all values are numeric
for col in X_test.columns:
    if pd.api.types.is_numeric_dtype(X_test[col]):
        X_test[col] = X_test[col].astype(float)

X_test.info()
X_train.info()


Before oversampling: (4088, 21)
Train Column 'gender_Female' contains non-numeric values.
Train Column 'gender_Male' contains non-numeric values.
Train Column 'gender_Other' contains non-numeric values.
Train Column 'ever_married_No' contains non-numeric values.
Train Column 'ever_married_Yes' contains non-numeric values.
Train Column 'work_type_Govt_job' contains non-numeric values.
Train Column 'work_type_Never_worked' contains non-numeric values.
Train Column 'work_type_Private' contains non-numeric values.
Train Column 'work_type_Self-employed' contains non-numeric values.
Train Column 'work_type_children' contains non-numeric values.
Train Column 'Residence_type_Rural' contains non-numeric values.
Train Column 'Residence_type_Urban' contains non-numeric values.
Train Column 'smoking_status_Unknown' contains non-numeric values.
Train Column 'smoking_status_formerly smoked' contains non-numeric values.
Train Column 'smoking_status_never smoked' contains non-numeric values.
Train Col

In [5]:
lr = LogisticRegression(random_state=2021, feature_names=X_train.columns, penalty='l1', solver='liblinear')
lr.fit(X_train, y_train)
print("Training finished.")

# %% Evaluate logistic regression model
y_pred = lr.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")

# %% Explain local prediction
lr_local = lr.explain_local(X_test[:100], y_test[:100], name='Logistic Regression')
show(lr_local)

Training finished.
F1 Score 0.5187432870725551
Accuracy 0.738747553816047


In [6]:
import pandas as pd
import numpy as np

# Train the Logistic Regression model (moved from cell qqSAqVymmAS7 to ensure 'lr' is defined)
lr = LogisticRegression(random_state=2021, feature_names=X_train.columns, penalty='l1', solver='liblinear')
lr.fit(X_train, y_train)
print("Logistic Regression Model Training finished.")

# Get feature names from the training data
feature_names = X_train.columns.tolist()

# --- Questionnaire Style Input ---
print("\nPlease answer the following questions for stroke prediction:")
new_record_data = {}

# --- Gender ---
gender_options = ['male', 'female', 'other']
while True:
    gender_input = input(f"What is your gender? ({'/'.join(g.capitalize() for g in gender_options)}): ").strip().lower()
    if gender_input in gender_options:
        for opt in gender_options:
            new_record_data[f'gender_{opt.capitalize()}'] = 1.0 if gender_input == opt else 0.0
        break
    else:
        print(f"Invalid input. Please enter one of: {', '.join(g.capitalize() for g in gender_options)}.")

# --- Ever Married ---
married_options = ['yes', 'no']
while True:
    married_input = input(f"Are you currently or have you ever been married? ({'/'.join(m.capitalize() for m in married_options)}): ").strip().lower()
    if married_input in married_options:
        new_record_data['ever_married_Yes'] = 1.0 if married_input == 'yes' else 0.0
        new_record_data['ever_married_No'] = 1.0 if married_input == 'no' else 0.0
        break
    else:
        print(f"Invalid input. Please enter 'Yes' or 'No'.")

# --- Work Type ---
# Mapping user-friendly inputs to exact column names
work_type_options_map = {
    'govt job': 'work_type_Govt_job',
    'never worked': 'work_type_Never_worked',
    'private': 'work_type_Private',
    'self employed': 'work_type_Self-employed',
    'children': 'work_type_children'
}
while True:
    work_type_input = input(f"What is your work type? ({'/'.join(k.title() for k in work_type_options_map.keys())}): ").strip().lower()
    if work_type_input in work_type_options_map:
        selected_col = work_type_options_map[work_type_input]
        for col in work_type_options_map.values():
            new_record_data[col] = 1.0 if col == selected_col else 0.0
        break
    else:
        print(f"Invalid input. Please choose from: {', '.join(k.title() for k in work_type_options_map.keys())}.")

# --- Residence Type ---
residence_options = ['rural', 'urban']
while True:
    residence_input = input(f"What is your residence type? ({'/'.join(r.capitalize() for r in residence_options)}): ").strip().lower()
    if residence_input in residence_options:
        for opt in residence_options:
            new_record_data[f'Residence_type_{opt.capitalize()}'] = 1.0 if residence_input == opt else 0.0
        break
    else:
        print(f"Invalid input. Please enter 'Rural' or 'Urban'.")

# --- Smoking Status ---
smoking_options_map = {
    'unknown': 'smoking_status_Unknown',
    'formerly smoked': 'smoking_status_formerly smoked',
    'never smoked': 'smoking_status_never smoked',
    'smokes': 'smoking_status_smokes'
}
while True:
    smoking_input = input(f"What is your smoking status? ({'/'.join(k.title() for k in smoking_options_map.keys())}): ").strip().lower()
    if smoking_input in smoking_options_map:
        selected_col = smoking_options_map[smoking_input]
        for col in smoking_options_map.values():
            new_record_data[col] = 1.0 if col == selected_col else 0.0
        break
    else:
        print(f"Invalid input. Please choose from: {', '.join(k.title() for k in smoking_options_map.keys())}.")

# --- Numerical Inputs ---
numerical_features_prompts = {
    'age': "What is your age?",
    'avg_glucose_level': "What is your average glucose level?",
    'bmi': "What is your BMI?"
}
for feature, prompt_text in numerical_features_prompts.items():
    while True:
        try:
            # Try to get a sample value from X_train if available for guidance
            sample_value = X_train[feature].iloc[0] if feature in X_train.columns else ''
            value = float(input(f"{prompt_text} (e.g., {sample_value}): ").strip())
            new_record_data[feature] = value
            break
        except ValueError:
            print("Invalid input. Please enter a numerical value.")

# --- Boolean Inputs ---
boolean_features_prompts = {
    'hypertension': "Do you have hypertension? (Yes/No): ",
    'heart_disease': "Do you have heart disease? (Yes/No): "
}
for feature, prompt_text in boolean_features_prompts.items():
    while True:
        yes_no_input = input(prompt_text).strip().lower()
        if yes_no_input == 'yes':
            new_record_data[feature] = 1.0
            break
        elif yes_no_input == 'no':
            new_record_data[feature] = 0.0
            break
        else:
            print("Invalid input. Please enter 'Yes' or 'No'.")

# Ensure all feature_names from X_train are in new_record_data, defaulting to 0.0 if not covered by questions
# This is crucial for one-hot encoded features where only one option was chosen.
for feature_col in feature_names:
    if feature_col not in new_record_data:
        new_record_data[feature_col] = 0.0

# Convert the dictionary to a DataFrame, ensuring column order matches X_train
# It's crucial to pass the index=[0] to create a single-row DataFrame
new_record_df = pd.DataFrame([new_record_data], columns=feature_names)

# Ensure all columns are float type, as per X_train/X_test preprocessing
for col in new_record_df.columns:
    if pd.api.types.is_numeric_dtype(new_record_df[col]):
        new_record_df[col] = new_record_df[col].astype(float)

print("\nNew record created:")
display(new_record_df)

# Predict the likelihood using the Logistic Regression model
# The model expects probabilities for binary classification (likelihood of being class 1)
predicted_likelihood = lr.predict_proba(new_record_df)[:, 1]

print(f"\nPredicted likelihood of stroke for the new record: {predicted_likelihood[0]*100:.2f}%")

# You can also get the class prediction (0 or 1)
predicted_class = lr.predict(new_record_df)
print(f"Predicted class (0=No Stroke, 1=Stroke): {predicted_class[0]}")

lr_local_new_record = lr.explain_local(new_record_df, predicted_class, name='Logistic Regression Local Explanation')
show(lr_local_new_record)

# Access the explanation data for the first instance
explanation_data = lr_local_new_record.data(0)
feature_contributions = dict(zip(explanation_data['names'], explanation_data['scores']))

# Find the feature with the highest absolute contribution
most_influential_feature = None
max_abs_contribution = 0

for feature, contribution in feature_contributions.items():
    if abs(contribution) > max_abs_contribution:
        max_abs_contribution = abs(contribution)
        most_influential_feature = feature

print(f"\nThe most influential factor for this prediction was: '{most_influential_feature}' with a contribution of {max_abs_contribution:.4f}")

print("\nAdvice: To ensure that the risk of getting a stroke is reduced, please ensure that: ")
print(" - you are exercising 20-30 minutes per day")
print(" - your BMI is between 18.5 and 27.5")
print(" - alcohol consumption is reduced")
print(" - blood pressure is regulated")
print(" - smoking habits are abandoned")
print("If you are concerned about your risk of having a stroke, please contact your local GP")


Logistic Regression Model Training finished.

Please answer the following questions for stroke prediction:
What is your gender? (Male/Female/Other): Male
Are you currently or have you ever been married? (Yes/No): Yes
What is your work type? (Govt Job/Never Worked/Private/Self Employed/Children): Government job
Invalid input. Please choose from: Govt Job, Never Worked, Private, Self Employed, Children.
What is your work type? (Govt Job/Never Worked/Private/Self Employed/Children): Govt Job
What is your residence type? (Rural/Urban): Rural
What is your smoking status? (Unknown/Formerly Smoked/Never Smoked/Smokes): Never Smoked
What is your age? (e.g., 53.0): 50
What is your average glucose level? (e.g., 175.92): 140
What is your BMI? (e.g., 26.9): 29
Do you have hypertension? (Yes/No): No
Do you have heart disease? (Yes/No): No

New record created:


,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,...,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,age,hypertension,heart_disease,avg_glucose_level,bmi
0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,50.0,0.0,0.0,140.0,29.0



Predicted likelihood of stroke for the new record: 23.28%
Predicted class (0=No Stroke, 1=Stroke): 0



The most influential factor for this prediction was: 'age' with a contribution of 4.3392

Advice: To ensure that the risk of getting a stroke is reduced, please ensure that: 
 - you are exercising 20-30 minutes per day
 - your BMI is between 18.5 and 27.5
 - alcohol consumption is reduced
 - blood pressure is regulated
 - smoking habits are abandoned
If you are concerned about your risk of having a stroke, please contact your local GP
